In [2]:
!pip install koeda konlpy
!pip install googletrans==4.0.0-rc1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 50.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [koeda]32m3/6 [konlpy]
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.9 MB/s eta 0:00:00
  DEPRECATION: Building 'googletrans' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'googletrans'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17455 sha256=f0

In [3]:
!sudo apt-get update
!sudo apt-get install -y openjdk-11-jdk

Get:1 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Get:3 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [3479 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble/multiverse amd64 Packages [331 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble/universe amd64 Packages [19.3 MB]
Get:8 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [34.2 kB]
Get:9 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1981 kB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1498 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble/main amd64 Packages [1808 kB]    
Get:12 http://archive.ubuntu.com/ubuntu noble/restricted amd64 Packages [117 kB]
Get:13 http://archive.ubuntu.com/ubuntu

In [4]:
import os
import pandas as pd
import random
import time
import re
from googletrans import Translator
from koeda import EDA

# 0. 환경 설정
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

# 1. 데이터 로드 및 컬럼명 변경 (가장 직관적인 방법)
df_mistakes = pd.read_csv('model_worst_mistakes.csv')
df_original = pd.read_csv('../Xuxeong/final_train_data.csv')

# ★ 핵심: 처음에 바로 '문장'을 'conversation'으로 변경
df_mistakes.rename(columns={'문장': 'conversation'}, inplace=True)

# 2. 라벨 전처리 (0, 1, 2, 3 추출)
def extract_label(text):
    match = re.search(r'\((\d+)\)', str(text))
    if match: return int(match.group(1))
    match = re.search(r'\d+', str(text))
    return int(match.group()) if match else -1

df_mistakes['label'] = df_mistakes['정답'].apply(extract_label)
df_target = df_mistakes[df_mistakes['label'].isin([0, 1, 2, 3])].copy()

# 3. 증강 함수 정의
def random_swap(sentence, n=2):
    words = str(sentence).split()
    if len(words) < 2: return sentence
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(words)), 2)
        words[idx1], words[idx2] = words[idx2], words[idx1]
    return ' '.join(words)

translator = Translator()
def back_translate(text, lang='en'):
    try:
        en = translator.translate(str(text), src='ko', dest=lang).text
        ko = translator.translate(en, src=lang, dest='ko').text
        time.sleep(1.2) # API 차단 방지
        return ko
    except: return text

eda = EDA(morpheme_analyzer="Okt", alpha_sr=0.2, alpha_ri=0.0, alpha_rs=0.0, prob_rd=0.0)
def synonym_replacement(text):
    try:
        res = eda(str(text))
        return res[0] if isinstance(res, list) else res
    except: return text

# 4. 증강 및 원본 병합
print("데이터 증강 및 병합을 시작합니다...")

# --- [1] RS (어순 변화) 5배수 ---
rs_list = []
for _ in range(5):
    temp = df_target.copy()
    temp['conversation'] = temp['conversation'].apply(lambda x: random_swap(x, n=2))
    rs_list.append(temp[['conversation', 'label']])

df_rs = pd.concat(rs_list, ignore_index=True)
final_rs = pd.concat([df_original, df_rs], ignore_index=True)
final_rs.to_csv('train_augmented_RS.csv', index=False, encoding='utf-8-sig')
print("RS 5배수 데이터셋 생성 완료")

# --- [2] SR (유의어 교체) 5배수 ---
sr_list = []
for _ in range(5):
    temp = df_target.copy()
    temp['conversation'] = temp['conversation'].apply(synonym_replacement)
    sr_list.append(temp[['conversation', 'label']])

df_sr = pd.concat(sr_list, ignore_index=True)
final_sr = pd.concat([df_original, df_sr], ignore_index=True)
final_sr.to_csv('train_augmented_SR.csv', index=False, encoding='utf-8-sig')
print("SR 5배수 데이터셋 생성 완료")

# --- [3] BT (역번역) 1배수 (안전하게 영어만 1회) ---
temp_bt = df_target.copy()
print("역번역(BT) 1배수 진행 중... (안정성을 위해 약 1~2분 소요됩니다)")
temp_bt['conversation'] = temp_bt['conversation'].apply(lambda x: back_translate(x, lang='en'))

final_bt = pd.concat([df_original, temp_bt[['conversation', 'label']]], ignore_index=True)
final_bt.to_csv('train_augmented_BT.csv', index=False, encoding='utf-8-sig')
print("BT 1배수 데이터셋 생성 완료")

print("모든 작업이 깔끔하게 끝났습니다!")

데이터 증강 및 병합을 시작합니다...
RS 5배수 데이터셋 생성 완료
SR 5배수 데이터셋 생성 완료
역번역(BT) 1배수 진행 중... (안정성을 위해 약 1~2분 소요됩니다)
BT 1배수 데이터셋 생성 완료
모든 작업이 깔끔하게 끝났습니다!


In [2]:
import pandas as pd

# 1. 저장된 데이터 로드
df_original = pd.read_csv('../Xuxeong/final_train_data.csv')
df_rs = pd.read_csv('train_augmented_RS.csv')
df_sr = pd.read_csv('train_augmented_SR.csv')
df_bt = pd.read_csv('train_augmented_BT.csv')

# 2. 개수 계산
orig_count = len(df_original)

# 3. 결과 출력
print("=== 📊 데이터 증강 결과 요약 ===")
print(f"▶ 원본 학습 데이터: {orig_count}개\n")

# RS (5배수)
rs_aug = len(df_rs) - orig_count
print(f"[RS (어순 변화 5배수)]")
print(f"총 {len(df_rs)}개 (원본 + {rs_aug}개 순수 증강)\n")

# SR (5배수)
sr_aug = len(df_sr) - orig_count
print(f"[SR (유의어 교체 5배수)]")
print(f"총 {len(df_sr)}개 (원본 + {sr_aug}개 순수 증강)\n")

# BT (1배수)
bt_aug = len(df_bt) - orig_count
print(f"[BT (역번역 1배수)]")
print(f"총 {len(df_bt)}개 (원본 + {bt_aug}개 순수 증강)")

=== 📊 데이터 증강 결과 요약 ===
▶ 원본 학습 데이터: 4740개

[RS (어순 변화 5배수)]
총 5150개 (원본 + 410개 순수 증강)

[SR (유의어 교체 5배수)]
총 5150개 (원본 + 410개 순수 증강)

[BT (역번역 1배수)]
총 4822개 (원본 + 82개 순수 증강)


#### 데이터 합치기

In [5]:
import pandas as pd

# 1. 파일 로드 (오타 수정 및 기준 데이터 정의)
df_train = pd.read_csv('./data/train_augmented.csv') # 기존에 쓰던 학습 데이터 (df_original 역할)
df_val = pd.read_csv('./data/validation_unseen.csv') # ⚠️ 검증셋 (병합에서 제외하여 누수 방지)
df_rs_full = pd.read_csv('train_augmented_RS.csv')
df_sr_full = pd.read_csv('train_augmented_SR.csv')
df_bt_full = pd.read_csv('train_augmented_BT.csv')

# 2. 각 파일에서 '증강된 부분'만 추출 (원본 데이터 이후의 행들)
# df_train을 원본 데이터(orig_len)의 기준으로 삼습니다.
orig_len = len(df_train)

df_rs_only = df_rs_full.iloc[orig_len:]
df_sr_only = df_sr_full.iloc[orig_len:]
df_bt_only = df_bt_full.iloc[orig_len:]

# 3. 하나의 데이터로 병합 (원본 + RS증강 + SR증강 + BT증강)
# ignore_index=True를 통해 인덱스를 0부터 다시 깨끗하게 정렬합니다.
df_final_all = pd.concat([df_train, df_rs_only, df_sr_only, df_bt_only], ignore_index=True)

# 4. 최종 저장 (한글 깨짐 방지를 위해 utf-8-sig 사용)
df_final_all.to_csv('train_augmented_ALL.csv', index=False, encoding='utf-8-sig')

# 5. 결과 리포트
print("=== 🚀 통합 데이터셋 생성 완료 ===")
print(f"1. 기준 원본 데이터 개수: {orig_len}개")
print(f"2. 추가된 증강 데이터: {len(df_rs_only) + len(df_sr_only) + len(df_bt_only)}개")
print(f"   - RS(+{len(df_rs_only)}), SR(+{len(df_sr_only)}), BT(+{len(df_bt_only)})")
print(f"3. 최종 통합 데이터 개수: {len(df_final_all)}개")
print(f"\n✅ 파일이 'train_augmented_ALL.csv'로 성공적으로 저장되었습니다.")

=== 🚀 통합 데이터셋 생성 완료 ===
1. 기준 원본 데이터 개수: 5642개
2. 추가된 증강 데이터: 0개
   - RS(+0), SR(+0), BT(+0)
3. 최종 통합 데이터 개수: 5642개

✅ 파일이 'train_augmented_ALL.csv'로 성공적으로 저장되었습니다.
